# Etapa 2: Pré-Processamento, Encoding e Escalonamento dos Dados

## 1. Contexto e Estratégia de Preparação de Dados

Este notebook realiza a transformação estruturada dos dados brutos consolidados na **Tabela Mestra** para o formato numérico otimizado para os algoritmos de Machine Learning.

### Diretrizes de Engenharia de Dados:
1. **Divisão Temporal Cega (*Time-Series Holdout*):** O dataset de teste é estritamente posterior ao treino, sem embaralhamento aleatório (*random shuffle*), prevenindo qualquer vazamento de dados do futuro.
2. **Tratamento de Categóricas (*Encoding*):**
   * Estações (11 categorias): One-Hot Encoding para preservar a independência linear de cada ambiente físico.
   * Plataforma (P1/P2): Mapeamento binário.
3. **Escalonamento Robusto (*RobustScaler*):** Normalização baseada em medianas e quartis ($IQR$), imune a outliers causados por picos de alarmes em falhas atípicas. O scaler é ajustado (*fitted*) estritamente nos dados de treino.



In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas com sucesso!")


✅ Bibliotecas carregadas com sucesso!


## 2. Ingestão da Tabela Mestra Bruta


In [2]:
path_mestra = "../../data/tabela_mestra_bruta.csv"
df = pd.read_csv(path_mestra)
df['data_ref'] = pd.to_datetime(df['data_ref'])

print(f"✅ Tabela Mestra carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas.")
print(f"Período total coberto: {df['data_ref'].min().strftime('%Y-%m-%d')} a {df['data_ref'].max().strftime('%Y-%m-%d')}")



✅ Tabela Mestra carregada: 607,530 linhas x 25 colunas.
Período total coberto: 2023-01-01 a 2026-02-28


## 3. Divisão Temporal Cega (Treino vs Teste)


In [3]:
# Split Temporal Cego: Treino (2023 a 2025) vs Teste (2026)
data_corte = pd.to_datetime('2026-01-01')

df_treino = df[df['data_ref'] < data_corte].copy()
df_teste = df[df['data_ref'] >= data_corte].copy()

print(f"Treino: {len(df_treino):,} linhas (até {df_treino['data_ref'].max().strftime('%Y-%m-%d')}) | Prevalência Target: {df_treino['target_falha_7d'].mean()*100:.2f}%")
print(f"Teste:  {len(df_teste):,} linhas ({df_teste['data_ref'].min().strftime('%Y-%m-%d')} a {df_teste['data_ref'].max().strftime('%Y-%m-%d')}) | Prevalência Target: {df_teste['target_falha_7d'].mean()*100:.2f}%")



Treino: 576,496 linhas (até 2025-12-31) | Prevalência Target: 8.50%
Teste:  31,034 linhas (2026-01-01 a 2026-02-28) | Prevalência Target: 8.30%


## 4. Encodings Estruturais (Estações e Plataforma)


In [4]:
def aplicar_encodings(df_in):
    df_out = df_in.copy()
    df_out['plataforma_bin'] = df_out['plataforma'].map({'P1': 0, 'P2': 1}).fillna(0).astype(int)
    df_out = pd.get_dummies(df_out, columns=['estacao'], prefix='estacao', drop_first=False)
    return df_out

df_treino_enc = aplicar_encodings(df_treino)
df_teste_enc = aplicar_encodings(df_teste)

# Alinhamento de colunas para garantir consistência
df_treino_enc, df_teste_enc = df_treino_enc.align(df_teste_enc, join='left', axis=1, fill_value=0)

cols_estacoes = [c for c in df_treino_enc.columns if c.startswith('estacao_')]
print(f"✅ Encodings concluídos! Estações codificadas ({len(cols_estacoes)}): {cols_estacoes}")



✅ Encodings concluídos! Estações codificadas (11): ['estacao_BUT', 'estacao_FAL', 'estacao_FRA', 'estacao_FRE', 'estacao_HIG', 'estacao_LUZ', 'estacao_MBI', 'estacao_PIN', 'estacao_PTA', 'estacao_REP', 'estacao_VSO']


## 5. Escalonamento Robusto com RobustScaler


In [5]:
cols_para_escalonar = [
    'idade_ativo_anos', 'porta',
    'dias_desde_ultima_zm01', 'dias_desde_ultima_zm02', 'dias_desde_ultima_zm03',
    'qtd_alarmes_1d', 'qtd_alarmes_7d', 'qtd_alarmes_14d', 'aceleracao_alarmes_7d',
    'qtd_anomalias_15s_7d', 'qtd_anomalias_30s_14d',
    'qtd_rce_7d', 'qtd_fo_7d', 'qtd_lsd_7d',
    'tempo_parada_acumulado_7d', 'max_tempo_normalizacao_7d',
    'tempo_medio_por_alarme_7d', 'proporcao_alarmes_graves_7d',
    'fator_risco_ciclo_zm01'
]

scaler = RobustScaler()
scaler.fit(df_treino_enc[cols_para_escalonar])

cols_scaled = [f"{c}_scaled" for c in cols_para_escalonar]
df_treino_enc[cols_scaled] = scaler.transform(df_treino_enc[cols_para_escalonar])
df_teste_enc[cols_scaled] = scaler.transform(df_teste_enc[cols_para_escalonar])

features = ['plataforma_bin', 'faixa_risco_zm01'] + cols_estacoes + cols_scaled
target_col = 'target_falha_7d'

print(f"✅ Total de features finais de modelagem: {len(features)}")



✅ Total de features finais de modelagem: 32


## 6. Exportação dos Datasets Pré-Processados e Artefatos


In [6]:
os.makedirs("../../data/modelos_artefatos", exist_ok=True)

cols_export = ['id_ativo', 'data_ref'] + features + [target_col]

df_treino_enc[cols_export].to_csv("../../data/treino_pronto_modelagem_v2.csv", index=False)
df_teste_enc[cols_export].to_csv("../../data/teste_pronto_modelagem_v2.csv", index=False)

joblib.dump(scaler, "../../data/modelos_artefatos/robust_scaler.joblib")

print("🎉 Datasets prontos para modelagem salvos com sucesso!")
print(f"   -> Treino: {df_treino_enc.shape[0]:,} amostras")
print(f"   -> Teste:  {df_teste_enc.shape[0]:,} amostras")
print(f"   -> Scaler salvo em data/modelos_artefatos/robust_scaler.joblib")



🎉 Datasets prontos para modelagem salvos com sucesso!
   -> Treino: 576,496 amostras
   -> Teste:  31,034 amostras
   -> Scaler salvo em data/modelos_artefatos/robust_scaler.joblib
